In [1]:
########################################################################################
#      PREPROCESSING — Pour Topic Modeling (TF-IDF)   :   environnement python torch310
########################################################################################

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
stopwords_eng = set(stopwords.words("english"))

# Chargement dataset
df = pd.read_csv("train.csv", header=None, names=["label", "title", "text"])

# Concaténation titre + texte
df["text"] = df["title"].fillna("") + " " + df["text"].fillna("")

# Nettoyage simple
def clean_for_topics(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_clean"] = df["text"].apply(clean_for_topics)

# Stopwords
df["text_clean"] = df["text_clean"].apply(
    lambda t: " ".join([w for w in t.split() if w not in stopwords_eng])
)

# Sous-échantillonnage raisonnable (25k lignes)
train_df_filtered = df.sample(25000, random_state=42).reset_index(drop=True)

print("Taille preprocessing topic modeling :", len(train_df_filtered))


[nltk_data] Downloading package stopwords to
[nltk_data]     D:\Users\utheza\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Taille preprocessing topic modeling : 25000


In [2]:
#############################################################
#        TOPIC MODELING — TF-IDF + KMeans (Baseline)
#############################################################

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# On prend toutes les lignes disponibles (≈ 25 000)
df_topics = train_df_filtered.copy()

# Vectorisation TF-IDF
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words="english"
)

X_tfidf = vectorizer.fit_transform(df_topics["text"])

print("Shape TF-IDF :", X_tfidf.shape)

# KMeans
k = 6   # nombre de thèmes approximatif
kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
labels = kmeans.fit_predict(X_tfidf)

df_topics["cluster"] = labels

# Silhouette Score
silhouette = silhouette_score(X_tfidf, labels)
print("Silhouette score :", silhouette)

# Top mots par cluster
terms = vectorizer.get_feature_names_out()

for i in range(k):
    centroid = kmeans.cluster_centers_[i]
    top10 = centroid.argsort()[-10:][::-1]
    print(f"\n===== Thème {i} =====")
    print([terms[j] for j in top10])



Shape TF-IDF : (25000, 50000)
Silhouette score : 0.003681563931244472

===== Thème 0 =====
['money', 'work', 'buy', 'waste', 'don', 'waste money', 'product', 'bought', 'time', 'battery']

===== Thème 1 =====
['cd', 'album', 'music', 'songs', 'song', 'like', 'great', 'good', 'best', 'listen']

===== Thème 2 =====
['book', 'read', 'books', 'reading', 'good', 'great', 'author', 'story', 'like', 'read book']

===== Thème 3 =====
['movie', 'movies', 'good', 'great', 'watch', 'film', 'like', 'acting', 'just', 'time']

===== Thème 4 =====
['product', 'great', 'use', 'good', 'price', 'quality', 'works', 'just', 'easy', 'used']

===== Thème 5 =====
['game', 'like', 'dvd', 'good', 'just', 'great', 'story', 'film', 'love', 'really']


In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

In [2]:
#############################################################
#      TOPIC MODELING — Sentence-BERT + KMeans (Avancé)
#############################################################

import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

#############################################################
# 1 — Chargement & preprocessing léger (adapté SBERT)
#############################################################

# Chargement dataset
df = pd.read_csv("train.csv", header=None, names=["label", "title", "text"])

# Concaténation titre + texte
df["text"] = (df["title"].fillna("") + " " + df["text"].fillna("")).astype(str)

# Nettoyage minimal (pas de stopwords, pas de lemmatisation)
def clean_for_sbert(text):
    text = re.sub(r"</?[a-zA-Z][^>]*>", " ", text)  # suppression HTML
    text = re.sub(r"\s+", " ", text).strip()        # espaces multiples
    return text

df["text_sbert"] = df["text"].apply(clean_for_sbert)

# Sous-échantillonnage pour rester fluide
df_topics = df.sample(20000, random_state=42).reset_index(drop=True)

print("Taille échantillon SBERT :", len(df_topics))

#############################################################
# 2 — Encodage Sentence-BERT
#############################################################

model_sbert = SentenceTransformer("all-MiniLM-L6-v2")  # rapide et robuste

embeddings = model_sbert.encode(
    df_topics["text_sbert"].tolist(),
    show_progress_bar=True
)

print("Shape SBERT embeddings :", embeddings.shape)

#############################################################
# 3 — Clustering KMeans
#############################################################

k = 6
kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
labels = kmeans.fit_predict(embeddings)

df_topics["cluster"] = labels

# Silhouette score
silhouette = silhouette_score(embeddings, labels)
print("Silhouette SBERT :", silhouette)

#############################################################
# 4 — Interprétation qualitative des clusters
#############################################################

for c in range(k):
    print(f"\n===== EXEMPLES CLUSTER {c} =====")
    print(
        df_topics[df_topics["cluster"] == c]["text_sbert"]
        .head(5)
        .tolist()
    )



Taille échantillon SBERT : 20000


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Shape SBERT embeddings : (20000, 384)
Silhouette SBERT : 0.047902241349220276

===== EXEMPLES CLUSTER 0 =====
['Toast too dark Even on the lowest setting, the toast is too dark for my liking. Also, the on light stays lit so you have to unplug it to avoid wasting electricity. Not the quality I expected from Cuisinart.', 'Not worth your time Might as well just use a knife, this product holds next to nothing. I found it to be a useless product.', "Garbage The handle broke clean off after TWO WEEKS. Not the plastic part, actually at the weld where the metal meets the pan. Wasn't overheated, wasn't abused, used 4 times and broke.", 'Great for making dough I ordered this bread machine for a vacation home that we own. I have a Cuisinart at my year-round home and it just has too many bells and whistles that I never use - and cost a lot more. The Sunbeam makes a great dough and was a great price.', "Definitely works Have used this and it definitely gets u going to the potty!! Bye bye water weig

In [18]:
# Prédire le thème d'avis rédigés par nous

test_reviews = [
    "I bought this product hoping it would solve a simple everyday problem, but the experience was disappointing. The device feels cheaply made and does not perform as expected. After only a few uses, the battery started losing charge very quickly, which makes it impractical for daily use. Considering the price, I honestly feel like I wasted my money and expected much better quality.",
    "This album is a great listen from start to finish. The sound quality is excellent and the production really highlights both the vocals and the instruments. A few songs stand out more than the others, but overall the track list is well balanced. I have been listening to it repeatedly and would definitely recommend it to music lovers.",    
    "I enjoyed reading this book, even though it was not perfect. The story is engaging and easy to follow, and the author does a good job developing the main characters. Some chapters felt a bit slow, but overall the writing style kept me interested until the end. It is a solid read for anyone who enjoys character-driven stories.",
    "The movie has strong acting performances and a well-constructed story, but it drags in a few places. Some scenes could have been shorter to keep a better pace. That said, the cinematography is impressive and several moments are genuinely memorable. Overall, it is an enjoyable film, even if it does not fully live up to its potential.",
    "This product is easy to use and does exactly what it promises. The quality feels solid and it works well for everyday use. I have been using it regularly for a few weeks now without any issues. For the price, it offers good value and seems durable enough to last.",    
    "Installation was more complicated than expected and the instructions were not very clear. I also ran into some compatibility issues at first, which was frustrating. Once everything was finally set up, the device worked fine, but the initial experience definitely affected my overall impression.",    
    "This game is a lot of fun to play and offers several hours of entertainment. The gameplay is simple to understand, but still engaging enough to keep things interesting. The controls are responsive, the levels are well designed, and there is a good balance between challenge and enjoyment. While the graphics are not groundbreaking, they fit the style of the game well. Overall, it is an enjoyable experience and a solid choice if you are looking for a game to relax and have fun with.",    
]


# Encodage Sentence-BERT
test_embeddings = model_sbert.encode(
    test_reviews,
    show_progress_bar=False
)

# Association d'un label à chaque Cluster
cluster_labels = {
    0: "Product performance & value perception",
    1: "Entertainment and leisure products",
    2: "Movies, documentaries and audiovisual content",
    3: "Music albums & CDs",
    4: "Books & literature",
    5: "Technology & accessories"
}

# Predictions
predicted_clusters = kmeans.predict(test_embeddings)

# Affichage avec label métier
for review, cluster in zip(test_reviews, predicted_clusters):
    label = cluster_labels.get(cluster, "Unknown theme")
    print("\n", review)
    print(f"------> Theme prédit: {label}")


 I bought this product hoping it would solve a simple everyday problem, but the experience was disappointing. The device feels cheaply made and does not perform as expected. After only a few uses, the battery started losing charge very quickly, which makes it impractical for daily use. Considering the price, I honestly feel like I wasted my money and expected much better quality.
------> Theme prédit: Technology & accessories

 This album is a great listen from start to finish. The sound quality is excellent and the production really highlights both the vocals and the instruments. A few songs stand out more than the others, but overall the track list is well balanced. I have been listening to it repeatedly and would definitely recommend it to music lovers.
------> Theme prédit: Music albums & CDs

 I enjoyed reading this book, even though it was not perfect. The story is engaging and easy to follow, and the author does a good job developing the main characters. Some chapters felt a bi

In [ ]:
# Sauvegarde du modèle sentenceBert

model_sbert.save("models/sentence_bert")

cluster_labels = {
    0: "Product performance & value perception",
    1: "Entertainment and leisure products",
    2: "Movies, documentaries and audiovisual content",
    3: "Music albums & CDs",
    4: "Books & literature",
    5: "Technology & accessories"
}

import joblib

joblib.dump(kmeans, "models/kmeans_topics.pkl")
joblib.dump(cluster_labels, "models/cluster_labels.pkl")  

['models/cluster_labels.pkl']

In [3]:
# Sauvegarde du modèle TF-IDF + Kmeans

import joblib
from pathlib import Path

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# 1) Sauvegarde du vectorizer TF-IDF
joblib.dump(vectorizer, MODELS_DIR / "tfidf_vectorizer_topics.pkl")

# 2) Sauvegarde du KMeans
joblib.dump(kmeans, MODELS_DIR / "kmeans_tfidf_topics.pkl")

# 3) Labels métiers des clusters (à adapter si besoin)
cluster_labels_tfidf = {
    0: "Customer dissatisfaction & product issues",
    1: "MMusic albums & CDs",
    2: "Books & literature",
    3: "Movies & cinema",
    4: "General product usage",
    5: "Video games & DVDs"
}

joblib.dump(cluster_labels_tfidf, MODELS_DIR / "cluster_labels_tfidf.pkl")

print("TF-IDF + KMeans saved successfully.")

TF-IDF + KMeans saved successfully.
